In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option('display.max_rows', None)

In [98]:
X_train = pd.read_csv('../../../data/processed/for_linear_model/DF,MF/predictors_train_filtered.csv')
X_test = pd.read_csv('../../../data/processed/for_linear_model/DF,MF/predictors_test_filtered.csv')
y_train = pd.read_csv('../../../data/processed/for_linear_model/DF,MF/target_train_filtered_logariphmed.csv')
y_test = pd.read_csv('../../../data/processed/for_linear_model/DF,MF/target_test_filtered_logariphmed.csv')

In [4]:
train_players = pd.read_csv('../../../data/processed/for_linear_model/DF,MF/train_players_names.csv')
test_players = pd.read_csv('../../../data/processed/for_linear_model/DF,MF/test_players_names.csv')

In [6]:
all_cols = X_train.columns
all_cols

Index(['Age', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK',
       'PKatt', 'CrdY', 'CrdR', 'Gls.1', 'Ast.1', 'G+A.1', 'G-PK.1', 'G+A-PK',
       '2CrdY', 'Fls', 'Fld', 'Off', 'Crs', 'Int', 'TklW', 'OG', 'Sh', 'SoT',
       'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1'],
      dtype='object')

## Регрессор

In [104]:
LR_regressor = LinearRegression()
LR_regressor.fit(X_train, y_train)

LinearRegression()

## Предсказанные значения

In [105]:
y_pred_log = LR_regressor.predict(X_test)
y_pred = np.exp(y_pred_log)
y_test_original = np.exp(y_test)

y_train_pred = np.exp(LR_regressor.predict(X_train))
y_train_original = np.exp(y_train)

## Метрики

In [106]:
mae_test = mean_absolute_error(y_test_original, y_pred)
rmse_test = np.sqrt(mean_squared_error(y_test_original, y_pred))
r2_test = r2_score(y_test_original, y_pred)
mape_test = mean_absolute_percentage_error(y_test_original, y_pred)

mae_train = mean_absolute_error(y_train_original, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train_original, y_train_pred))
r2_train = r2_score(y_train_original, y_train_pred)
mape_train = mean_absolute_percentage_error(y_train_original, y_train_pred)

metrics = pd.DataFrame({
    'MAE':[mae_test, mae_train],
    'RMSE':[rmse_test, rmse_train],
    'MAPE':[mape_test, mape_train],
    'R2':[r2_test, r2_train]
}, index = ['test', 'train'])

## Остатки

In [107]:
y_pred = pd.Series(np.reshape(y_pred, len(y_pred)), name='Value_pred')
y_train_pred = pd.Series(np.reshape(y_train_pred, len(y_train_pred)), name='Value_pred')

comp_leftovers_test = pd.concat([test_players, y_pred, y_test_original], axis=1)
comp_leftovers_train = pd.concat([train_players, y_train_pred, y_train_original], axis=1)

## Коэффициенты

In [108]:
coeffs = LR_regressor.coef_[0]
coeffs_df = pd.DataFrame({
    'coeff':coeffs.astype('float')
}, index=X_train.columns)
coeffs_df = coeffs_df.sort_values(by='coeff', key=abs, ascending=False)

# Все переменные

In [13]:
coeffs_df

,coeff
Min,8.684038
90s,-6.710016
league_GB1,0.775071
Starts,-0.763686
Ast.1,0.541095
MP,-0.534376
SoT,0.513828
Sh,-0.373773
Age,-0.350489
G+A-PK,-0.332881


In [14]:
metrics

,MAE,RMSE,MAPE,R2
test,6.413018,13.968448,2.493828,0.172109
train,2.504211,4.562421,0.708134,0.641897


In [15]:
comp_leftovers_train

,Player,Value_pred,Value
0,Claudio Falcao,1.466037,0.80
1,Josha Vagnoman,6.367078,10.00
2,Samuele Birindelli,7.460604,2.80
3,Fernando Ferreira Fonseca,0.818566,0.50
4,Jairo Riedewald,0.828498,1.80
5,Jackson Tchatchoua,15.504043,8.00
6,Reece James,15.548355,30.00
7,Ismael Doukoure,14.895639,18.00
8,Mauro Junior,4.745205,12.00
9,Miroslav Bogosavac,0.611272,1.00


In [16]:
comp_leftovers_test

,Player,Value_pred,Value
0,Danil Stepanov,0.561864,0.60
1,Eric Garcia,7.871803,18.00
2,Leandro Bacuna,14.795869,0.30
3,Ridwane M'Barki,0.251768,0.25
4,Nicolas Cozza,6.160999,3.50
5,Nikolai Rasskazov,1.412038,1.50
6,Raphael Guerreiro,5.609009,8.00
7,Nilton Varela,1.718796,1.50
8,Filippo Terracciano,1.799802,3.20
9,Julius Dirksen,1.722087,0.40


# Потенциальные переменные для добавления и удаления

## Текущие столбцы

In [97]:
base_cols = ['Age', '90s', 'Gls', 'Ast', 'G+A',
       '2CrdY', 'G+A.1', 'G-PK.1',
       'Crs', 'Int',
       'SoT%', 'G/Sh', 'G/SoT', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']

## Для добавления:

In [99]:
cols_to_add = [col for col in all_cols if not(col in base_cols)]
new_cols = pd.DataFrame({
    'mae':[mae_test],
    'mape':[mape_test],
    'r2':[r2_test]
}, index=['base'])
for col in cols_to_add:
    X_train_new = X_train[base_cols + [col]]
    X_test_new = X_test[base_cols + [col]]
    LR_regressor_new = LinearRegression()
    LR_regressor_new.fit(X_train_new, y_train)
    y_pred_new = np.exp(LR_regressor_new.predict(X_test_new))
    mae_new = mean_absolute_error(y_pred_new, y_test_original)
    mape_new = mean_absolute_percentage_error(y_pred_new, y_test_original)
    r2_new = r2_score(y_pred_new, y_test_original)
    if mae_new<mae_test or mape_new<mape_test or r2_new > r2_test:
        row = pd.DataFrame([[mae_new, mape_new, r2_new]], columns=new_cols.columns, index=[col])
        new_cols = pd.concat([new_cols, row])
new_cols

,mae,mape,r2
base,5.168998,1.080572,0.228252
Starts,5.157480,1.197914,-3.239179


## Для удаления:

In [100]:
cols_to_delete = [col for col in base_cols if not(col.startswith('league'))]
del_cols = pd.DataFrame({
    'mae':[mae_test],
    'mape':[mape_test],
    'r2':[r2_test]
}, index=['base'])
for col in cols_to_delete:
    X_train_new = X_train[[c for c in base_cols if c != col]]
    X_test_new = X_test[[c for c in base_cols if c != col]]
    LR_regressor_new = LinearRegression()
    LR_regressor_new.fit(X_train_new, y_train)
    y_pred_new = np.exp(LR_regressor_new.predict(X_test_new))
    mae_new = mean_absolute_error(y_pred_new, y_test_original)
    mape_new = mean_absolute_percentage_error(y_pred_new, y_test_original)
    r2_new = r2_score(y_pred_new, y_test_original)
    if mae_new < mae_test or mape_new < mape_test or r2_new > r2_test:
        row = pd.DataFrame([[mae_new, mape_new, r2_new]], columns=del_cols.columns, index=[col])
        del_cols = pd.concat([del_cols, row])
del_cols

,mae,mape,r2
base,5.168998,1.080572,0.228252
Gls,5.168998,1.196868,-3.243716
Ast,5.168998,1.196868,-3.243716
G+A,5.168998,1.196868,-3.243716
G-PK.1,5.167895,1.196433,-3.268970


# Модель 1

In [25]:
X_train = X_train[['Age', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK',
       'PKatt', 'CrdR', 'Gls.1', 'G+A.1', 'G-PK.1', 'G+A-PK',
        'Fls', 'Fld', 'Off', 'Crs', 'Int', 'TklW',
       'SoT%', 'SoT/90', 'G/Sh', 'G/SoT', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['Age', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK',
       'PKatt', 'CrdR', 'Gls.1', 'G+A.1', 'G-PK.1', 'G+A-PK',
        'Fls', 'Fld', 'Off', 'Crs', 'Int', 'TklW',
       'SoT%', 'SoT/90', 'G/Sh', 'G/SoT', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [31]:
coeffs_df

,coeff
league_GB1,0.696683
90s,0.666750
G+A-PK,-0.621583
G+A.1,0.535955
Age,-0.367677
league_L1,0.263304
G/Sh,0.262555
league_ES1,0.260256
league_FR1,0.252451
league_IT1,0.229571


In [32]:
metrics

,MAE,RMSE,MAPE,R2
test,5.583333,13.414909,1.320670,0.236424
train,2.780889,5.064896,0.752939,0.558675


# Модель 2

In [37]:
X_train = X_train[['Age', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK',
       'PKatt', '2CrdY', 'CrdR', 'Gls.1', 'G+A.1', 'G-PK.1', 'G+A-PK',
       'Fld', 'Off', 'Crs', 'Int', 'TklW',
       'SoT%', 'G/Sh', 'G/SoT', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['Age', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK',
       'PKatt', '2CrdY', 'CrdR', 'Gls.1', 'G+A.1', 'G-PK.1', 'G+A-PK',
       'Fld', 'Off', 'Crs', 'Int', 'TklW',
       'SoT%', 'G/Sh', 'G/SoT', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [43]:
coeffs_df

,coeff
league_GB1,0.676696
90s,0.539723
G+A-PK,-0.522506
G+A.1,0.428700
Age,-0.387808
league_FR1,0.253876
league_ES1,0.252338
league_L1,0.248153
league_IT1,0.230948
G/Sh,0.208503


In [44]:
metrics

,MAE,RMSE,MAPE,R2
test,5.294987,13.486479,1.113241,0.228255
train,2.824135,5.152226,0.767549,0.543325


In [46]:
comp_leftovers_train

,Player,Value_pred,Value
0,Claudio Falcao,1.615323,0.80
1,Josha Vagnoman,4.861826,10.00
2,Samuele Birindelli,5.296176,2.80
3,Fernando Ferreira Fonseca,0.865093,0.50
4,Jairo Riedewald,0.733491,1.80
5,Jackson Tchatchoua,16.180793,8.00
6,Reece James,13.595550,30.00
7,Ismael Doukoure,11.039145,18.00
8,Mauro Junior,3.795261,12.00
9,Miroslav Bogosavac,0.637725,1.00


In [47]:
comp_leftovers_test

,Player,Value_pred,Value
0,Danil Stepanov,0.673700,0.60
1,Eric Garcia,8.439097,18.00
2,Leandro Bacuna,3.408636,0.30
3,Ridwane M'Barki,0.318025,0.25
4,Nicolas Cozza,4.191708,3.50
5,Nikolai Rasskazov,1.932328,1.50
6,Raphael Guerreiro,4.322365,8.00
7,Nilton Varela,1.873128,1.50
8,Filippo Terracciano,2.597621,3.20
9,Julius Dirksen,1.197083,0.40


# Модель 3

In [52]:
X_train = X_train[['Age', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK',
       'PKatt', '2CrdY', 'CrdR', 'G+A.1', 'G-PK.1', 'G+A-PK',
       'Fld', 'Off', 'Crs', 'Int', 'TklW',
       'SoT%', 'G/Sh', 'G/SoT', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['Age', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK',
       'PKatt', '2CrdY', 'CrdR', 'G+A.1', 'G-PK.1', 'G+A-PK',
       'Fld', 'Off', 'Crs', 'Int', 'TklW',
       'SoT%', 'G/Sh', 'G/SoT', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [58]:
coeffs_df

,coeff
league_GB1,0.674901
G+A-PK,-0.605878
90s,0.532636
G+A.1,0.511992
Age,-0.388645
league_FR1,0.255534
league_ES1,0.251850
league_L1,0.246298
league_IT1,0.229492
G/Sh,0.214313


In [59]:
metrics

,MAE,RMSE,MAPE,R2
test,5.276381,13.471834,1.123086,0.229930
train,2.827389,5.162423,0.768464,0.541516


In [60]:
comp_leftovers_train

,Player,Value_pred,Value
0,Claudio Falcao,1.607933,0.80
1,Josha Vagnoman,4.868975,10.00
2,Samuele Birindelli,5.371044,2.80
3,Fernando Ferreira Fonseca,0.864382,0.50
4,Jairo Riedewald,0.738891,1.80
5,Jackson Tchatchoua,16.070223,8.00
6,Reece James,13.594247,30.00
7,Ismael Doukoure,11.234434,18.00
8,Mauro Junior,3.756748,12.00
9,Miroslav Bogosavac,0.639687,1.00


In [61]:
comp_leftovers_test

,Player,Value_pred,Value
0,Danil Stepanov,0.673258,0.60
1,Eric Garcia,8.522207,18.00
2,Leandro Bacuna,3.467723,0.30
3,Ridwane M'Barki,0.320286,0.25
4,Nicolas Cozza,4.231772,3.50
5,Nikolai Rasskazov,1.906790,1.50
6,Raphael Guerreiro,4.358680,8.00
7,Nilton Varela,1.855384,1.50
8,Filippo Terracciano,2.608296,3.20
9,Julius Dirksen,1.214354,0.40


# Модель 4

In [62]:
X_train = X_train[['Age', '90s', 'Gls', 'Ast', 'G+A', 'PK',
       '2CrdY', 'G+A.1', 'G-PK.1',
       'Off',
       'SoT%', 'G/Sh', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['Age', '90s', 'Gls', 'Ast', 'G+A', 'PK',
       '2CrdY', 'G+A.1', 'G-PK.1',
       'Off',
       'SoT%', 'G/Sh', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [68]:
coeffs_df

,coeff
league_GB1,0.704861
Age,-0.389401
90s,0.387681
league_FR1,0.298692
league_L1,0.296181
league_ES1,0.293833
league_IT1,0.269400
G/Sh,0.142935
Ast,0.122717
SoT%,0.108856


In [69]:
metrics

,MAE,RMSE,MAPE,R2
test,5.374772,13.687768,1.255729,0.205046
train,2.725314,5.027361,0.766593,0.565192


# Модель 5

In [74]:
X_train = X_train[['Age', '90s', 'Gls', 'Ast', 'G+A',
       '2CrdY', 'G+A.1', 'G-PK.1',
       'Off', 'Crs', 'Int',
       'SoT%', 'G/Sh', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['Age', '90s', 'Gls', 'Ast', 'G+A',
       '2CrdY', 'G+A.1', 'G-PK.1',
       'Off', 'Crs', 'Int',
       'SoT%', 'G/Sh', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [80]:
coeffs_df

,coeff
league_GB1,0.684653
90s,0.476487
Age,-0.375656
league_L1,0.272964
league_FR1,0.266198
league_ES1,0.258095
league_IT1,0.237932
G/Sh,0.139745
G+A,0.116520
Ast,0.111290


In [81]:
metrics

,MAE,RMSE,MAPE,R2
test,5.299735,13.488795,1.024503,0.227990
train,2.749328,5.088813,0.768807,0.554498


In [87]:
comp_leftovers_train

,Player,Value_pred,Value
0,Claudio Falcao,1.371245,0.80
1,Josha Vagnoman,4.985697,10.00
2,Samuele Birindelli,5.189832,2.80
3,Fernando Ferreira Fonseca,0.922994,0.50
4,Jairo Riedewald,0.718637,1.80
5,Jackson Tchatchoua,17.078961,8.00
6,Reece James,14.026476,30.00
7,Ismael Doukoure,13.509108,18.00
8,Mauro Junior,3.439411,12.00
9,Miroslav Bogosavac,0.667824,1.00


In [88]:
comp_leftovers_test

,Player,Value_pred,Value
0,Danil Stepanov,0.656877,0.60
1,Eric Garcia,8.140353,18.00
2,Leandro Bacuna,2.233194,0.30
3,Ridwane M'Barki,0.321154,0.25
4,Nicolas Cozza,4.274569,3.50
5,Nikolai Rasskazov,1.648817,1.50
6,Raphael Guerreiro,4.416489,8.00
7,Nilton Varela,1.946334,1.50
8,Filippo Terracciano,2.549692,3.20
9,Julius Dirksen,1.209188,0.40


# Модель 6

In [89]:
X_train = X_train[['Age', '90s', 'Gls', 'Ast', 'G+A',
       '2CrdY', 'G+A.1', 'G-PK.1',
       'Crs', 'Int',
       'SoT%', 'G/Sh', 'G/SoT', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['Age', '90s', 'Gls', 'Ast', 'G+A',
       '2CrdY', 'G+A.1', 'G-PK.1',
       'Crs', 'Int',
       'SoT%', 'G/Sh', 'G/SoT', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [95]:
coeffs_df

,coeff
league_GB1,0.687271
90s,0.514902
Age,-0.379389
league_L1,0.271412
league_FR1,0.267954
league_ES1,0.259541
league_IT1,0.247741
G/Sh,0.210644
G+A,0.126646
G/SoT,-0.121998


In [96]:
metrics

,MAE,RMSE,MAPE,R2
test,5.168998,13.486506,1.080572,0.228252
train,2.791708,5.149400,0.768641,0.543826


In [102]:
comp_leftovers_train

,Player,Value_pred,Value
0,Claudio Falcao,1.468495,0.80
1,Josha Vagnoman,5.300171,10.00
2,Samuele Birindelli,5.200596,2.80
3,Fernando Ferreira Fonseca,0.973234,0.50
4,Jairo Riedewald,0.707242,1.80
5,Jackson Tchatchoua,13.267947,8.00
6,Reece James,13.788649,30.00
7,Ismael Doukoure,11.489177,18.00
8,Mauro Junior,3.452216,12.00
9,Miroslav Bogosavac,0.640042,1.00


In [103]:
comp_leftovers_test

,Player,Value_pred,Value
0,Danil Stepanov,0.676700,0.60
1,Eric Garcia,8.712923,18.00
2,Leandro Bacuna,2.613380,0.30
3,Ridwane M'Barki,0.326678,0.25
4,Nicolas Cozza,4.183914,3.50
5,Nikolai Rasskazov,1.679672,1.50
6,Raphael Guerreiro,5.077972,8.00
7,Nilton Varela,2.171921,1.50
8,Filippo Terracciano,2.567638,3.20
9,Julius Dirksen,1.186583,0.40


# Модель 7

In [101]:
X_train = X_train[['Age', '90s', 'Gls', 'Ast',
       '2CrdY',
       'Crs', 'Int',
       'SoT%', 'G/Sh', 'G/SoT', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['Age', '90s', 'Gls', 'Ast',
       '2CrdY',
       'Crs', 'Int',
       'SoT%', 'G/Sh', 'G/SoT', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [109]:
coeffs_df

,coeff
league_GB1,0.687696
90s,0.542010
Age,-0.384654
league_L1,0.275449
league_FR1,0.268338
league_ES1,0.260469
league_IT1,0.250034
G/Sh,0.197610
Ast,0.156486
Gls,0.130956


In [110]:
metrics

,MAE,RMSE,MAPE,R2
test,5.226608,13.506199,1.046830,0.225996
train,2.759821,5.120070,0.765459,0.549008


In [111]:
comp_leftovers_train

,Player,Value_pred,Value
0,Claudio Falcao,1.522152,0.80
1,Josha Vagnoman,5.369468,10.00
2,Samuele Birindelli,5.194542,2.80
3,Fernando Ferreira Fonseca,0.988770,0.50
4,Jairo Riedewald,0.689526,1.80
5,Jackson Tchatchoua,13.000959,8.00
6,Reece James,13.839613,30.00
7,Ismael Doukoure,11.104437,18.00
8,Mauro Junior,3.344609,12.00
9,Miroslav Bogosavac,0.625462,1.00


In [112]:
comp_leftovers_test

,Player,Value_pred,Value
0,Danil Stepanov,0.662416,0.60
1,Eric Garcia,8.728282,18.00
2,Leandro Bacuna,2.459301,0.30
3,Ridwane M'Barki,0.308934,0.25
4,Nicolas Cozza,4.206646,3.50
5,Nikolai Rasskazov,1.683859,1.50
6,Raphael Guerreiro,5.090845,8.00
7,Nilton Varela,2.203839,1.50
8,Filippo Terracciano,2.513474,3.20
9,Julius Dirksen,1.110591,0.40
